In [1]:
import geopandas as gpd

In [2]:
BD = gpd.read_file("bgd_admin2.geojson")

In [3]:
BD

,adm2_name,adm2_name1,adm2_name2,adm2_name3,adm2_pcode,adm1_name,adm1_name1,adm1_name2,adm1_name3,adm1_pcode,...,area_sqkm,version,lang,lang1,lang2,lang3,adm2_ref_name,center_lat,center_lon,geometry
0,Barguna,None,None,None,BD1004,Barishal,None,None,None,BD10,...,1534.961669,v03,en,None,None,None,None,22.128262,90.110133,"POLYGON ((90.23712 22.48259, 90.2365 22.48261,..."
1,Barishal,None,None,None,BD1006,Barishal,None,None,None,BD10,...,2553.415382,v03,en,None,None,None,None,22.819387,90.368787,"POLYGON ((90.54483 23.08013, 90.53161 23.08028..."
2,Bhola,None,None,None,BD1009,Barishal,None,None,None,BD10,...,3471.662936,v03,en,None,None,None,None,22.310321,90.763983,"POLYGON ((90.71614 22.86847, 90.71563 22.86847..."
3,Jhalokati,None,None,None,BD1042,Barishal,None,None,None,BD10,...,735.705149,v03,en,None,None,None,None,22.572341,90.181898,"POLYGON ((90.19507 22.78471, 90.19495 22.78518..."
4,Patuakhali,None,None,None,BD1078,Barishal,None,None,None,BD10,...,3125.033859,v03,en,None,None,None,None,22.165187,90.407257,"POLYGON ((90.55692 22.59322, 90.55642 22.59348..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,Thakurgaon,None,None,None,BD5594,Rangpur,None,None,None,BD55,...,1805.624951,v03,en,None,None,None,None,25.990096,88.344545,"POLYGON ((88.31877 26.20356, 88.31861 26.20371..."
60,Habiganj,None,None,None,BD6036,Sylhet,None,None,None,BD60,...,2610.697376,v03,en,None,None,None,None,24.369432,91.431707,"POLYGON ((91.3895 24.68954, 91.38892 24.69008,..."
61,Moulvibazar,None,None,None,BD6058,Sylhet,None,None,None,BD60,...,2673.509020,v03,en,None,None,None,None,24.481077,91.916467,"POLYGON ((92.25007 24.83285, 92.2501 24.83288,..."
62,Sunamganj,None,None,None,BD6090,Sylhet,None,None,None,BD60,...,3666.856262,v03,en,None,None,None,None,24.939849,91.345596,"POLYGON ((91.27023 25.20411, 91.27015 25.2042,..."


In [8]:
Khulna=BD[BD["adm2_name"]=="Khulna"]

In [9]:
Khulna

,adm2_name,adm2_name1,adm2_name2,adm2_name3,adm2_pcode,adm1_name,adm1_name1,adm1_name2,adm1_name3,adm1_pcode,...,area_sqkm,version,lang,lang1,lang2,lang3,adm2_ref_name,center_lat,center_lon,geometry
34,Khulna,None,None,None,BD4047,Khulna,None,None,None,BD40,...,4457.26501,v03,en,None,None,None,None,22.365739,89.452816,"POLYGON ((89.45595 23.01135, 89.45484 23.01171..."


In [10]:
Khulna.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [11]:
Khulna_utm = Khulna.to_crs(epsg=32646)

In [12]:
Khulna_utm.crs

<Projected CRS: EPSG:32646>
Name: WGS 84 / UTM zone 46N
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Between 90°E and 96°E, northern hemisphere between equator and 84°N, onshore and offshore. Bangladesh. Bhutan. China. Indonesia. Mongolia. Myanmar (Burma). Russian Federation.
- bounds: (90.0, 0.0, 96.0, 84.0)
Coordinate Operation:
- name: UTM zone 46N
- method: Transverse Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [13]:
Khulna_utm["area_km2"] = Khulna_utm.area / 1e6
Khulna_utm["area_km2"]

34    4468.420856
Name: area_km2, dtype: float64

In [14]:
Khulna_utm.to_file("khulna_boundary_utm.geojson", driver="GeoJSON")

In [15]:
Khulna.to_file("khulna_boundary_wgs84.geojson", driver="GeoJSON")

In [16]:
Khulna_utm.total_bounds

array([ 112339.54837027, 2399384.06008307,  167234.75950518,
       2549219.87053913])

In [17]:
import geopandas as gpd
from shapely.geometry import box
import numpy as np

xmin, ymin, xmax, ymax = Khulna_utm.total_bounds

grid_size = 500  # meter

cells = []

for x in np.arange(xmin, xmax, grid_size):
    for y in np.arange(ymin, ymax, grid_size):
        cells.append(box(x, y, x + grid_size, y + grid_size))

grid = gpd.GeoDataFrame(
    {"grid_id": range(len(cells))},
    geometry=cells,
    crs=Khulna_utm.crs
)

# Khulna boundary এর ভিতরের grid only রাখো
grid = gpd.overlay(grid, Khulna_utm, how="intersection")

grid.head()

,grid_id,adm2_name,adm2_name1,adm2_name2,adm2_name3,adm2_pcode,adm1_name,adm1_name1,adm1_name2,adm1_name3,...,version,lang,lang1,lang2,lang3,adm2_ref_name,center_lat,center_lon,area_km2,geometry
0,158,Khulna,None,None,None,BD4047,Khulna,None,None,None,...,v03,en,None,None,None,None,22.365739,89.452816,4468.420856,"POLYGON ((112839.548 2478884.06, 112839.548 24..."
1,159,Khulna,None,None,None,BD4047,Khulna,None,None,None,...,v03,en,None,None,None,None,22.365739,89.452816,4468.420856,"POLYGON ((112839.548 2479384.06, 112839.548 24..."
2,160,Khulna,None,None,None,BD4047,Khulna,None,None,None,...,v03,en,None,None,None,None,22.365739,89.452816,4468.420856,"POLYGON ((112839.548 2479884.06, 112839.548 24..."
3,161,Khulna,None,None,None,BD4047,Khulna,None,None,None,...,v03,en,None,None,None,None,22.365739,89.452816,4468.420856,"POLYGON ((112839.548 2479884.06, 112590.393 24..."
4,197,Khulna,None,None,None,BD4047,Khulna,None,None,None,...,v03,en,None,None,None,None,22.365739,89.452816,4468.420856,"POLYGON ((112839.548 2498384.06, 112839.548 24..."


In [18]:
grid["grid_area_m2"] = grid.area
grid["grid_area_km2"] = grid["grid_area_m2"] / 1e6

In [22]:
grid.columns

Index(['grid_id', 'adm2_name', 'adm2_name1', 'adm2_name2', 'adm2_name3',
       'adm2_pcode', 'adm1_name', 'adm1_name1', 'adm1_name2', 'adm1_name3',
       'adm1_pcode', 'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3',
       'adm0_pcode', 'valid_on', 'valid_to', 'area_sqkm', 'version', 'lang',
       'lang1', 'lang2', 'lang3', 'adm2_ref_name', 'center_lat', 'center_lon',
       'area_km2', 'geometry', 'grid_area_m2', 'grid_area_km2'],
      dtype='str')

In [23]:
grid.to_file("khulna_500m_grid_utm.geojson", driver="GeoJSON")

In [24]:
grid_wgs84 = grid.to_crs(epsg=4326)
grid_wgs84.to_file("khulna_500m_grid_wgs84.geojson", driver="GeoJSON")